In [2]:
!pip install implicit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 52.8 MB/s eta 0:00:0000:0100:01


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import scipy.sparse as sp

from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import bm25_weight

In [15]:
FACTORS = 64
REGULARIZATION = 0.01
ITERATIONS = 20
USE_GPU = False

INPUT_DIR = Path('/kaggle/input/notebooks/artembez/als-data-preparing')
OUTPUT_DIR = Path('/kaggle/working/')

### Загрузим данные

In [9]:
train_matrix = sp.load_npz(INPUT_DIR / 'train_matrix.npz')
test_matrix = sp.load_npz(INPUT_DIR / 'test_matrix.npz')

train_matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 92644 stored elements and shape (26244, 200)>

### Применим BM25 для ограничения влияния популярных продуктов

In [11]:
train_matrix_weighted = bm25_weight(train_matrix, K1 = 100, B = 0.7)
train_matrix_csr = train_matrix_weighted.tocsr()
train_matrix_csr

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 92644 stored elements and shape (26244, 200)>

### Тренируем модель

In [13]:
model = AlternatingLeastSquares(
    factors = FACTORS,
    regularization = REGULARIZATION,
    iterations = ITERATIONS,
    use_gpu = USE_GPU,
    random_state = 42
)

model.fit(train_matrix_csr, show_progress = True)

/usr/local/lib/python3.12/dist-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/20 [00:00<?, ?it/s]

In [16]:
import pickle

with open(OUTPUT_DIR / 'als_model.pkl', 'wb') as f:
    pickle.dump(model, f)